In [ ]:
# Example 1: Basic Function Optimization
from zoopt import Dimension, Objective, Parameter, Opt
import numpy as np

# Define a simple quadratic function to minimize
def quadratic_function(x):
    return (x.get_x()[0] - 2)**2 + (x.get_x()[1] + 1)**2 + 3

# Set up the optimization problem
dim = Dimension(2, [[-5, 5], [-5, 5]], [True, True])  # 2D space with bounds
objective = Objective(quadratic_function, dim)

# Configure optimization parameters
parameter = Parameter(budget=100, sequential=True)

# Perform optimization
solution = Opt.min(objective, parameter)

# Extract results
optimal_x = solution.get_x()
optimal_value = solution.get_value()

print(f"ZOOpt Basic Optimization Results:")
print(f"Optimal point: x = {optimal_x}")
print(f"Optimal value: f(x) = {optimal_value:.6f}")
print(f"True optimum: x = [2, -1], f(x) = 3")
print(f"Distance from true optimum: {np.sqrt((optimal_x[0] - 2)**2 + (optimal_x[1] + 1)**2):.6f}")

# Verify optimization quality
improvement = abs(optimal_value - 3.0)
print(f"Optimization error: {improvement:.6f}")

## Example 2: Multi-Dimensional Rastrigin Function Optimization

In [ ]:
# Example 2: Multi-Dimensional Rastrigin Function Optimization
import math

# Define the challenging Rastrigin function (many local minima)
def rastrigin_function(x):
    A = 10
    n = len(x.get_x())
    result = A * n
    for xi in x.get_x():
        result += xi**2 - A * math.cos(2 * math.pi * xi)
    return result

# Set up high-dimensional optimization problem
dimensions = 5
bounds = [[-5.12, 5.12] for _ in range(dimensions)]
is_continuous = [True] * dimensions

dim = Dimension(dimensions, bounds, is_continuous)
objective = Objective(rastrigin_function, dim)

# Use more budget for this challenging problem
parameter = Parameter(budget=500, sequential=True)

# Perform optimization
solution = Opt.min(objective, parameter)

# Analyze results
optimal_x = solution.get_x()
optimal_value = solution.get_value()
global_optimum = 0.0  # Rastrigin global minimum is 0 at x = [0, 0, ..., 0]

print(f"Rastrigin Function Optimization ({dimensions}D):")
print(f"Optimal point: {[round(x, 4) for x in optimal_x]}")
print(f"Optimal value: {optimal_value:.6f}")
print(f"Global optimum value: {global_optimum}")
print(f"Distance from global optimum: {optimal_value - global_optimum:.6f}")

# Calculate distance from true global minimum
euclidean_distance = np.sqrt(sum(xi**2 for xi in optimal_x))
print(f"Euclidean distance from origin: {euclidean_distance:.6f}")

# Check solution quality
if optimal_value < 1.0:
    print("✅ Good optimization result (close to global minimum)")
else:
    print("⚠️ Challenging function - found local minimum")

## Example 3: Mixed-Type Optimization (Continuous and Discrete)

In [ ]:
# Example 3: Mixed-Type Optimization (Continuous and Discrete)
def mixed_function(x):
    # x[0], x[1]: continuous variables
    # x[2]: discrete variable (0, 1, 2, 3, 4)
    variables = x.get_x()
    continuous_part = (variables[0] - 1)**2 + (variables[1] + 0.5)**2
    discrete_penalty = abs(variables[2] - 2) * 0.5  # Best when discrete var = 2
    return continuous_part + discrete_penalty

# Define mixed-type dimension
# First two dimensions: continuous in [-3, 3]
# Third dimension: discrete with 5 possible values [0, 1, 2, 3, 4]
dim = Dimension(3, [[-3, 3], [-3, 3], [0, 4]], [True, True, False])
objective = Objective(mixed_function, dim)

# Optimize mixed-type problem
parameter = Parameter(budget=200, sequential=True)
solution = Opt.min(objective, parameter)

optimal_x = solution.get_x()
optimal_value = solution.get_value()

print(f"Mixed-Type Optimization Results:")
print(f"Optimal point: {optimal_x}")
print(f"  Continuous vars: [{optimal_x[0]:.4f}, {optimal_x[1]:.4f}]")
print(f"  Discrete var: {int(optimal_x[2])}")
print(f"Optimal value: {optimal_value:.6f}")

# True optimum analysis
true_optimum_continuous = [1.0, -0.5]
true_optimum_discrete = 2
true_minimum = 0.0

print(f"True optimum: continuous = {true_optimum_continuous}, discrete = {true_optimum_discrete}")
print(f"True minimum value: {true_minimum}")

# Check discrete variable optimization
if int(optimal_x[2]) == true_optimum_discrete:
    print("✅ Correct discrete variable found")
else:
    print(f"⚠️ Discrete variable: found {int(optimal_x[2])}, expected {true_optimum_discrete}")

# Check overall optimization quality
optimization_error = optimal_value - true_minimum
print(f"Optimization error: {optimization_error:.6f}")

## Example 4: Constrained Optimization with Penalty Methods

In [ ]:
# Example 4: Constrained Optimization with Penalty Methods
def constrained_function(x):
    # Objective: minimize (x[0] - 2)^2 + (x[1] - 1)^2
    # Subject to: x[0] + x[1] <= 2 (constraint)
    variables = x.get_x()
    objective_value = (variables[0] - 2)**2 + (variables[1] - 1)**2
    
    # Add penalty for constraint violation
    constraint_violation = max(0, variables[0] + variables[1] - 2)
    penalty = 1000 * constraint_violation**2  # Large penalty for violation
    
    return objective_value + penalty

# Set up constrained optimization
dim = Dimension(2, [[-2, 4], [-2, 4]], [True, True])
objective = Objective(constrained_function, dim)

# Run constrained optimization
parameter = Parameter(budget=300, sequential=True)
solution = Opt.min(objective, parameter)

optimal_x = solution.get_x()
optimal_value = solution.get_value()

print(f"Constrained Optimization Results:")
print(f"Optimal point: [{optimal_x[0]:.4f}, {optimal_x[1]:.4f}]")
print(f"Optimal value: {optimal_value:.6f}")

# Check constraint satisfaction
constraint_value = optimal_x[0] + optimal_x[1]
print(f"Constraint: x[0] + x[1] = {constraint_value:.4f} (should be ≤ 2)")

if constraint_value <= 2.001:  # Small tolerance for numerical errors
    print("✅ Constraint satisfied")
else:
    print("⚠️ Constraint violated")

# Compute true constrained minimum
# Lagrange multipliers show optimum at intersection: x[0] + x[1] = 2
# Combined with gradient condition gives optimum at [1.5, 0.5]
true_constrained_optimum = [1.5, 0.5]
true_constrained_value = (1.5 - 2)**2 + (0.5 - 1)**2

print(f"True constrained optimum: {true_constrained_optimum}")
print(f"True constrained value: {true_constrained_value:.6f}")

# Calculate optimization accuracy
distance_from_true = np.sqrt((optimal_x[0] - 1.5)**2 + (optimal_x[1] - 0.5)**2)
print(f"Distance from true optimum: {distance_from_true:.6f}")

## Example 5: Production Hyperparameter Tuning Pipeline

In [ ]:
# Example 5: Production Hyperparameter Tuning Pipeline
import time

# Simulate a machine learning model validation function
def ml_model_validation(x):
    # Hyperparameters: learning_rate, n_estimators, max_depth, regularization
    params = x.get_x()
    learning_rate = params[0]
    n_estimators = int(params[1])
    max_depth = int(params[2])
    regularization = params[3]
    
    # Simulate model training and validation (fast approximation)
    # Lower learning rate generally better but slower convergence
    lr_penalty = abs(learning_rate - 0.05) * 2
    
    # Too few or too many estimators are suboptimal
    estimator_penalty = abs(n_estimators - 100) / 100 * 0.5
    
    # Moderate depth usually optimal
    depth_penalty = abs(max_depth - 6) * 0.1
    
    # Regularization around 0.1 typically optimal
    reg_penalty = abs(regularization - 0.1) * 3
    
    # Add some noise to simulate real validation variance
    noise = np.random.normal(0, 0.01)
    
    # Validation error (lower is better)
    validation_error = 0.2 + lr_penalty + estimator_penalty + depth_penalty + reg_penalty + noise
    return max(0.05, validation_error)  # Ensure minimum realistic error

# Define hyperparameter search space
# [learning_rate, n_estimators, max_depth, regularization]
dim = Dimension(4, 
               [[0.001, 0.3],    # learning_rate: continuous
                [50, 200],       # n_estimators: discrete
                [3, 10],         # max_depth: discrete  
                [0.01, 0.5]],    # regularization: continuous
               [True, False, False, True])

objective = Objective(ml_model_validation, dim)

# Run hyperparameter optimization
print("Starting hyperparameter optimization...")
start_time = time.time()

parameter = Parameter(budget=150, sequential=True)
np.random.seed(42)  # For reproducible results
solution = Opt.min(objective, parameter)

optimization_time = time.time() - start_time

# Extract optimal hyperparameters
optimal_params = solution.get_x()
optimal_error = solution.get_value()

print(f"\nHyperparameter Optimization Results:")
print(f"Optimization time: {optimization_time:.2f} seconds")
print(f"Best validation error: {optimal_error:.6f}")
print(f"\nOptimal hyperparameters:")
print(f"  Learning rate: {optimal_params[0]:.6f}")
print(f"  N estimators: {int(optimal_params[1])}")
print(f"  Max depth: {int(optimal_params[2])}")
print(f"  Regularization: {optimal_params[3]:.6f}")

# Compare with typical defaults - create a proper mock object
class DefaultParams:
    def __init__(self, values):
        self.values = values
    def get_x(self):
        return self.values

default_params = DefaultParams([0.1, 100, 6, 0.1])
default_error = ml_model_validation(default_params)
improvement = (default_error - optimal_error) / default_error * 100

print(f"\nOptimization Performance:")
print(f"Default hyperparameters error: {default_error:.6f}")
print(f"Improvement over defaults: {improvement:.2f}%")
print("✅ Hyperparameter tuning complete - ready for production!")